# Iteration analysis

This notebook centralises the performance-evaluation workflow for the different model iterations. It pulls the metrics (RMSE and $R^2$) from each experiment's metadata and builds a comparative analysis: rankings and an interactive complexity-vs-error plot to pick the configuration that best trades off the number of features against accuracy.

## 1. Imports and initialisation

In [ ]:
import os
import glob
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Config
ITERATIONS_DIR = "" 
FILE_PATTERN = "*_meta.json"

## 2. Loading the iterations

In [ ]:
def load_experiments(folder):
    files = glob.glob(os.path.join(folder, FILE_PATTERN))
    print(f"Encontrados {len(files)} experimentos en '{folder}'...")
    
    data = []
    
    for file_path in files:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
                
            cfg = meta.get("cfg", {})
            metrics = meta.get("metrics_global", {})
            by_day = meta.get("metrics_by_day", [])
            
            # Basic info
            row = {
                "id": cfg.get("iteration_id", "unknown"),
                "short_id": meta.get("exp_id", "unknown")[:15], 
                "description": cfg.get("description", "").strip(),
                "model_depth": cfg.get("model_params", {}).get("max_depth", "?"),
                "n_features": len(meta.get("features", [])),
                "features_list": ", ".join(meta.get("features", [])),
                
                # Global metrics
                "RMSE_Global": metrics.get("rmse", 9999),
                "R2_Global": metrics.get("r2", 0),
            }
            
            # Metrics by day 
            for day_stat in by_day:
                date = day_stat["date"]
                row[f"RMSE_{date}"] = day_stat["rmse"]
                row[f"R2_{date}"] = day_stat["r2"]
                
            data.append(row)
            
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    return pd.DataFrame(data)

## 3. Analysis and visualisation

In [ ]:
def analyze_and_plot(df):
    if df.empty:
        print("No data to analyse.")
        return

    # Ordering by best global RMSE
    df = df.sort_values("RMSE_Global", ascending=True)
    
    print("\n" + "="*50)
    print("  BEST MODELS")
    print("="*50)
    
    best_global = df.iloc[0]
    print(f"\n🌍 BEST GLOBAL: {best_global['id']}")
    print(f"   RMSE: {best_global['RMSE_Global']:.2f} | R2: {best_global['R2_Global']:.3f}")
    print(f"   Features ({best_global['n_features']}): {best_global['features_list']}")
    
    day_cols = [c for c in df.columns if c.startswith("RMSE_20")]
    for col in day_cols:
        date = col.replace("RMSE_", "")
        best_day = df.sort_values(col, ascending=True).iloc[0]
        print(f"\n📅 BEST DAY {date}: {best_day['id']}")
        print(f"   RMSE: {best_day[col]:.2f}")

    # Comparison table
    print("\n" + "="*50)
    print("📊 COMPARISON TABLE")
    print("="*50)
    cols_show = ["id", "n_features", "model_depth", "RMSE_Global"] + day_cols
    print(df[cols_show].head(10).to_string(index=False))

    #  Plotly Scatter Plot
    df["hover_text"] = (
        "<b>" + df["id"] + "</b><br>" +
        "<i>" + df["description"] + "</i><br><br>" +
        "Features: " + df["features_list"]
    )

    fig = px.scatter(
        df,
        x="n_features",
        y="RMSE_Global",
        color="model_depth",  
        size="R2_Global",     
        hover_name="id",
        hover_data={
            "id": False, 
            "description": True, 
            "n_features": False,
            "RMSE_Global": ":.2f"
        },
        text="id",            
        title="Experiments: complexity vs error",
        labels={"n_features": "Number of features", "RMSE_Global": "Global error (RMSE)"},
        template="plotly_dark"
    )

    fig.update_traces(textposition='top center')
    fig.update_layout(height=600)
    
    
    fig.add_hline(y=df["RMSE_Global"].min(), line_dash="dash", line_color="green", annotation_text="Best global")

    
    fig.show()

if __name__ == "__main__":
    df = load_experiments(ITERATIONS_DIR)
    analyze_and_plot(df)